# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to an LLM provider (**MiniMax** in this copy) using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

> **My copy:** the course original uses Microsoft Foundry + `az login`. This copy runs on **MiniMax**
> (an OpenAI-compatible API) so no Azure subscription is needed.

Before running this notebook, make sure you have:

1. **A MiniMax API key** from the [MiniMax Platform](https://platform.minimaxi.com/) (with a small prepaid balance if required).
2. **A repo-root `.env` file** (git-ignored) containing:
   - `MINIMAX_API_KEY` - your API key.
   - `MINIMAX_BASE_URL` - optional, defaults to `https://api.minimax.io/v1`.
   - `MINIMAX_MODEL_ID` - optional, defaults to `MiniMax-M3`.
3. **The repo-root `.venv`** with `requirements.txt` installed (the first code cell is just a reminder; nothing is installed from the notebook).

The setup cell below imports `get_chat_client()` from `my-work/common/llm.py`, which builds the client from whichever provider key is set in `.env`.

In [1]:
# (deps are installed in the repo-root .venv from requirements.txt)


In [2]:
import sys
sys.path.insert(0, "../")  # lets this notebook import my-work/common/llm.py

from agent_framework import tool
from common.llm import get_chat_client

# Swapped from the course original: FoundryChatClient + AzureCliCredential -> provider-agnostic helper
provider = get_chat_client()


C:\Users\Balaji Calluru\Documents\Learning\AI\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\Balaji Calluru\Documents\Learning\AI\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


Using provider: minimax


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [3]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [4]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

I'll pull up the list of destinations we have available.<think>The user wants a warm beach destination. From the list, the best beach destinations with warm weather: Barcelona (Mediterranean, warm summers), Sydney (beaches but variable), Rio de Janeiro (warm beaches, iconic), Bali (tropical warm beaches), Cape Town (beaches, but cooler - Mediterranean-ish), Miami not listed.

Let me give recommendations with details.

Top picks: Rio de Janeiro (warm, tropical-ish, Copacabana/Ipanema), Bali (tropical, warm year-round, beaches), Barcelona (Mediterranean, warm summers, Barceloneta).

Let me present a few and ask about preferences.</think>

Great timing — several of our destinations fit the "warm beach" bill nicely. Here are the standouts:

**🏖️ Top Picks**

- **Rio de Janeiro, Brazil** — Copacabana and Ipanema are warm almost year-round (roughly 75–85°F), with dramatic mountain-and-ocean scenery, vibrant beach culture, and endless food and music.
- **Bali, Indonesia** — A true tropical es

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [5]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

I'll

 pull

 up the available

 destinations to give

 you the

 latest info on Tokyo.

... (58 outputs left out)

 **Seoul-style** city breaks, or **Sydney**? I'm

 also happy to help you shape the trip around specific

 interests (food, culture, nightlife, nature, family travel

).

## Summary

In this lesson you learned how to:

- **Create a provider** with `get_chat_client()`, which wraps the Microsoft Agent Framework's `OpenAIChatCompletionClient` and connects to MiniMax. (The course original uses `FoundryChatClient` with Azure CLI sign-in.)
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

### What changed from the course original

| Course original | This copy |
|---|---|
| `FoundryChatClient` + `AzureCliCredential` (`az login`) | `get_chat_client()` from `my-work/common/llm.py` |
| `AZURE_AI_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME` | `MINIMAX_API_KEY` (plus optional base URL and model ID) |
| Foundry-hosted model (e.g. `gpt-5-mini`) | MiniMax model (default `MiniMax-M3`) |

Notes from running it:

- **Chat Completions, not Responses:** `OpenAIChatClient` targets OpenAI's newer Responses API, which third-party providers may not offer, so the helper uses `OpenAIChatCompletionClient`. Its model argument is `model=` (the course's older sample uses `model_id=`, which this installed version rejects).
- **Visible reasoning:** MiniMax returns its reasoning inline as `<think>...</think>` blocks in the output above. That is fine for learning, but a customer-facing agent should strip them.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.